In [53]:
import igraph
from fa2_modified import ForceAtlas2
import random
import os
import pickle as pk

In [54]:
def describe(g):
    print( g.vcount() )
    print( g.ecount() )
    print( g.vertex_attributes() )
    print( g.edge_attributes() )

In [55]:
print("Available network graph files:")
for g in set( [g.split('_')[0] for g in os.listdir("graphs/")] ):
    print("-", g)

Available network graph files:
- LitLangMed
- Humanities


In [56]:
dataset = 'LitLangMed'

In [57]:
gc = igraph.Graph.Read_Pickle( 'graphs/' + dataset + '_cocitation_graph.pk' )
describe(gc)

5297
886977
['name', 'title', 'school', 'type', 'authors', 'continent', 'community']
['weight']


### Filter

In [58]:
filt_thresh = 5

In [59]:
# Delete edges with cocitation weight less than threshold
gc.es.select(weight_lt = filt_thresh).delete()

# Delete vertices with zero degree
gc.vs.select(_degree_lt=1).delete()

print( gc.vcount() )
print( gc.ecount() )

3628
15773


### Prepare for layout

In [60]:
color_by = 'continent' # community / continent / section / function / quality

In [62]:
# Create color palette for communities
unique_communities = sorted( list( set(gc.vs[color_by]) ) )
palette = igraph.ClusterColoringPalette( n=len( unique_communities ) )

In [63]:
# Color vertices

for v in gc.vs:   
    comm_i = unique_communities.index( v[color_by] )
    v['color'] = palette[comm_i]

# Store vertex strengths
vertex_strengths = gc.strength( weights=gc.es['weight'] )
for i, v in enumerate( gc.vs ):
    v['strength'] = vertex_strengths[i]
del vertex_strengths

### Run ForceAtlas2

In [79]:
random.seed(1327)

forceatlas2 = ForceAtlas2(
                        # Behavior alternatives
                        outboundAttractionDistribution=False,  # Dissuade hubs
                        linLogMode=False,  # NOT IMPLEMENTED
                        adjustSizes=False,  # Prevent overlap (NOT IMPLEMENTED)
                        edgeWeightInfluence=1,

                        # Performance
                        jitterTolerance=1.0,  # Tolerance
                        barnesHutOptimize=True,
                        barnesHutTheta=1.2,
                        multiThreaded=False,  # NOT IMPLEMENTED

                        # Tuning
                        scalingRatio=1.0,
                        strongGravityMode=True,
                        gravity=1,

                        # Log
                        verbose=True)

l = forceatlas2.forceatlas2_igraph_layout( gc, pos=None, iterations=1000, weight_attr='weight' )

100%|██████████████████████████████████████████████████████████| 1000/1000 [00:24<00:00, 41.46it/s]

BarnesHut Approximation  took  6.96  seconds
Repulsion forces  took  16.20  seconds
Gravitational forces  took  0.01  seconds
Attraction forces  took  0.09  seconds
AdjustSpeedAndApplyForces step  took  0.43  seconds


In [80]:
# Write layout
with open("layouts/" + dataset + "_" + str(filt_thresh) +  ".pkl", "wb") as f:
    pk.dump(l, f)

## Plot

In [64]:
with open("layouts/" + dataset + "_" + str(filt_thresh) + ".pkl", "rb") as f:
    l = pk.load(f)

In [65]:
igraph.plot(
    gc,
    'plots/' + dataset + "_" + str(filt_thresh) + '_cocitations.png',
    layout=l, 
    vertex_size = [(s/200)+3 for s in gc.vs['strength']],
    vertex_color = gc.vs['color'],
    vertex_label = [v['authors'] if v['strength'] >= 2000 else '' for v in gc.vs],
    vertex_label_color = 'black',
    vertex_label_size = [(s/250) for s in gc.vs['strength']],
    edge_width=0.15,
    bbox=(1000,1000),
    margin=20
)

# Get top authors in each community

In [12]:
import pandas as pd
from collections import defaultdict

In [24]:
n_authors = 20

In [25]:
authors_by_community = defaultdict(list)

for v in gc.vs:
    authors_by_community[v['community']].append((v['authors'], v['strength']))

In [26]:
top_authors_by_community = {}

for comm_id, authors in sorted( authors_by_community.items() ):
    
    authors_sorted = sorted(authors, key=lambda x: x[1], reverse=True)
    authors_sorted = [ au[0] + ', ' + str(au[1]) for au in authors_sorted]
    top_authors_by_community[comm_id] = authors_sorted[:n_authors]

In [27]:
top_authors_df = pd.DataFrame.from_dict(top_authors_by_community, orient='index').T
top_authors_df = top_authors_df.add_prefix('Community ')
top_authors_df

,Community 20,Community 21,Community 22,Community 23,Community 24,Community 25,Community 26,Community 27,Community 28,Community 29,Community 30,Community 31,Community 32,Community 33,Community 34,Community 35,Community 36
0,"P Bourdieu, 5595.0","Y Vera, 3367.0","L Cohen; L Manion; K Morrison, 38.0","B Latour, 406.0","N Brodie, 1570.0","L Gordon, 11625.0","Dbe, 3528.0","M Archer, 1988.0","J Cummins, 2702.0","S Tarp, 2065.0","S Pankhurst, 6244.0","B Head, 1771.0","R Mesthrie, 2916.0","Danai Mupotsa, 12.0","C Mutwa, 1336.0","C De Wet; L Jacobs, 24.0","John Tyson, 84.0"
1,"Sabc, 1786.0","R Muponde, 3367.0","E Guba; Y Lincoln, 36.0","Y Kazeem, 384.0","J Creswell, 300.0","See Gordon; L, 3960.0","N Hanna, 57.0","R Bhaskar, 1710.0","O García, 2600.0","D Geeraerts, 1776.0","N Cunard, 4059.0","C Mackenzie, 1275.0","E Mantzaris, 2435.0",None,"C Harrison, 180.0","C De Wet, 6.0","E James; Wyn, 66.0"
2,"R Yin, 660.0","J Butler, 2978.0","H Abadzi, 34.0","S Bradshaw; P Howard, 291.0","S Mathews, 141.0","S Ndlovu-Gatsheni, 3024.0","P Leedy; J Ormrod, 37.0","P Freire, 1535.0","E Gutt, 2534.0","M Alberts, 188.0","Comrade Mannin; Comrade, 4059.0","A Gagiano, 169.0","D Lewis, 648.0",None,"B Leshoai, 180.0","S Fox; L Stallworth, 6.0","S Kimbrough, 66.0"
3,"G Berger, 378.0","T Dangarembga, 2415.0","C Perfetti, 34.0","C Cadwalladr, 291.0","R Matzopoulos, 141.0","P Gqola, 2562.0","J Stommel, 37.0","L Shulman, 265.0","E Mokibelo, 2178.0","R Gouws, 188.0","E Mannin, 3616.0","D Driver, 126.0","L Janiszewski; E Alexakis, 412.0",None,"M Ani, 138.0",None,"Randy Maddox, 66.0"
4,"A Bryman, 349.0","K Muchemwa, 2415.0","N Spaull, 30.0","M Castells, 291.0","N, 141.0","G Daniels, 2421.0","E Babbie; J Mouton, 25.0","R Stake, 165.0","L Makalela, 382.0",S Chetty; T Isaac; H Manganye; N Mpondwana; L ...,"V Woolf, 2765.0","G Eilersen, 126.0","C Stewart, 412.0",None,"I Baraka, 138.0",None,"William Cowper, 26.0"
5,"C Arndt, 346.0","M Epprecht, 600.0","J Johnson, 25.0","R Diresta, 291.0","L Vetten, 141.0","H Wasserman, 2026.0","C Kivunja, 25.0","S Clegg, 144.0","L Nyati-Ramahobo, 280.0","R Gouws; D Prinsloo, 144.0","T Eliot, 2275.0","L.-S Beard, 42.0","M Thompson, 412.0",None,"R Kavanagh, 138.0",None,"David Stern, 26.0"
6,"N Garnham, 346.0","C Hove, 600.0","N Denzin; Y Lincoln, 23.0","N Egbunike, 291.0","A Altbeker, 48.0","S Biko, 2016.0","J Adler, 24.0","H Giroux, 144.0","O García; J Kleifgen, 233.0","S Nielsen, 144.0","J Joyce, 1060.0","R Blythe, 42.0","J Blommaert, 339.0",None,"J Mbiti, 62.0",None,"W Edited; Reginald Ward; Richard Heitzenrater,..."
7,"D Mcquail, 346.0","R Primorac, 600.0","D Mertens, 11.0","K Starbird, 291.0","K Breckenridge, 48.0","J Gordon, 2005.0","A Bates, 24.0","A Sayer, 144.0","H Batibo, 213.0","D Nkomo, 144.0","D Lawrence, 1060.0","M Cappelli, 42.0","G Baderoon, 312.0",None,"A Armah, 50.0",None,"William Williams, 26.0"
8,"S Cottle, 324.0","C Shaw, 525.0","V Braun; V Clarke; P Weate, 6.0","H Wasserman; D Madrid-Morales, 291.0","Yvonne Jewkes, 48.0","J Duncan, 1219.0","C Granberg; J Olsson, 24.0","S Badat, 86.0","A Chebanne, 213.0","S Khan, 140.0","J Rose, 810.0","G Castrillón, 42.0","U Dhupelia-Mesthrie, 312.0",None,"J De Graft, 50.0",None,"Richard Brantley, 26.0"
9,"J Keane, 324.0","C Achebe, 483.0","U, 6.0","S Woolley; P Howard, 291.0","J Klaehn; A Mullen, 48.0","N Couldry, 890.0","R Kozma, 24.0","J Jansen, 74.0","M Madiba, 213.0","K Heugh, 109.0","L Gordon; Nancy Cunard, 798.0","C Clayton, 42.0","G Lamb, 312.0",None,"F Dike, 50.0",None,"Richard Heitzenrater, 26.0"


In [28]:
top_authors_df.to_excel('xlsx/' + dataset + '_top_authors_by_community.xlsx', index=False)